# Zi Music Studio (free, Google Colab)

Generates a full song (music + lyrics) from a genre and an idea, then saves it to **Google Drive** in a RouteNote-ready folder.

**Before you start:** Runtime > Change runtime type > **T4 GPU** (free). Then run the cells in order. Cell 1 takes about 10-20 minutes the first time (downloads the model). Cell 2 is the app: pick a genre, press **Generate**.

Model: **ACE-Step 1.5** (MIT licence, commercial use allowed). No paid service, no card, no API key.

In [ ]:
#@title 1) Setup (run once per session)
import os, subprocess, sys, time, shutil, urllib.request, json

# GPU check
try:
    out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
except Exception:
    out = ""
if not out:
    raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", out)

OUT_ROOT = '/content/drive/MyDrive/Zi_Music'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive could not be mounted (", e, "). Saving to the Colab session instead: /content/Zi_Music (download it from the Files panel, it is lost when the session ends).")
    OUT_ROOT = '/content/Zi_Music'
os.makedirs(OUT_ROOT, exist_ok=True)

REPO = '/content/ACE-Step-1.5'
subprocess.run([sys.executable,"-m","pip","install","-q","uv"],check=True)
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable),"uv")
print("uv:", UV)
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","--depth","1","https://github.com/ace-step/ACE-Step-1.5",REPO],check=True)
print("Installing dependencies (several minutes)...")
# --- T4 fix: run the lyric encoder in fp32 (fp16 overflows -> NaN with lyrics) ---
_p = REPO + "/acestep/core/generation/handler/init_service_loader.py"
_s = open(_p).read()
if "_zi_fp32_lyric" not in _s:
    import re as _re
    _helper = '''

def _zi_fp32_lyric(model):
    import torch
    try:
        enc = model.encoder.lyric_encoder
        if next(enc.parameters()).dtype != torch.float16:
            return
    except Exception:
        return
    orig = enc.forward
    def _up(x):
        return x.float() if torch.is_tensor(x) and x.is_floating_point() else x
    def fwd(*a, **k):
        a = [_up(x) for x in a]
        k = {n: _up(x) for n, x in k.items()}
        enc.float()
        try:
            out = orig(*a, **k)
        finally:
            enc.half()
        if torch.is_tensor(out):
            return out.half()
        if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
            out.last_hidden_state = out.last_hidden_state.half()
            return out
        if isinstance(out, tuple):
            return tuple(x.half() if torch.is_tensor(x) and x.is_floating_point() else x for x in out)
        return out
    enc.forward = fwd
'''
    _m = _re.search(r"^([ \t]+)self\.model\.eval\(\)\n", _s, _re.M)
    assert _m, "eval() anchor not found"
    _s = _s[:_m.end()] + _m.group(1) + "_zi_fp32_lyric(self.model)\n" + _s[_m.end():] + _helper
    open(_p, "w").write(_s)
    print("Applied T4 fp32 lyric-encoder fix.")

r = subprocess.run([UV,"sync"],cwd=REPO,capture_output=True,text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit("Dependency install failed (see above).")

env = dict(os.environ,
    ACESTEP_API_HOST='127.0.0.1', ACESTEP_API_PORT='8001',
    ACESTEP_CONFIG_PATH='acestep-v15-turbo',
    ACESTEP_LM_MODEL_PATH='acestep-5Hz-lm-0.6B',
    ACESTEP_LM_BACKEND='pt', ACESTEP_INIT_LLM='true',
    ACESTEP_USE_FLASH_ATTENTION='false', MPLBACKEND='Agg')
log = open('/content/acestep.log','w')
SERVER = subprocess.Popen([UV,"run","acestep-api"],cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT)

BASE = 'http://127.0.0.1:8001'
print("Starting the music model and downloading weights (first time: 5-15 min)...")
ready = False
for i in range(240):
    if SERVER.poll() is not None: break
    try:
        with urllib.request.urlopen(BASE+'/health', timeout=5) as f:
            if f.status == 200: ready = True; break
    except Exception:
        pass
    time.sleep(5)
if not ready:
    print(open('/content/acestep.log').read()[-4000:])
    raise SystemExit("The music server did not start (log above). Send this log to your assistant.")
print("Ready. Run cell 2.")

In [ ]:
#@title 2) Studio: pick genre, press Generate
import json, re, time, csv, datetime, subprocess, urllib.request, urllib.error, random
import ipywidgets as w
from IPython.display import display, clear_output, Audio, HTML, Image as IPImage

GENRES = {
 "Techno":  "driving techno, 130 bpm, hypnotic analog synth, punchy kick, rolling bassline, dark club atmosphere, sparse whispered vocal",
 "R&B":     "smooth modern R&B, 85 bpm, warm electric piano, soft drums, silky bass, emotional soulful vocal with harmonies",
 "Hip-Hop": "hip-hop, 92 bpm, hard 808 bass, crisp trap hi-hats, dark piano loop, confident rap vocal, strong flow",
 "Pop":     "catchy modern pop, 110 bpm, bright synths, punchy drums, big singable chorus, polished female vocal",
}
PALETTES = {
 "Techno":  [(5,5,20),(70,0,140),(0,200,255)],
 "R&B":     [(30,10,40),(170,60,110),(255,190,120)],
 "Hip-Hop": [(15,15,15),(120,90,20),(240,200,60)],
 "Pop":     [(255,90,170),(255,200,80),(90,200,255)],
}

genre = w.Dropdown(options=list(GENRES), value="Pop", description="Genre")
idea = w.Text(value="a late night drive through the city", description="Idea", layout=w.Layout(width="520px"))
lang = w.Dropdown(options=[("English","en"),("Arabic","ar"),("Spanish","es"),("French","fr")], value="en", description="Lyrics")
dur = w.IntSlider(value=150, min=60, max=240, step=10, description="Seconds")
instr = w.Checkbox(value=False, description="Instrumental (no vocals/lyrics)")
title_in = w.Text(value="", placeholder="leave empty = auto title", description="Title", layout=w.Layout(width="520px"))
artist = w.Text(value="", placeholder="your artist name", description="Artist", layout=w.Layout(width="520px"))
writer = w.Text(value="", placeholder="your legal first + last name (composer/lyricist)", description="Writer", layout=w.Layout(width="520px"))
btn = w.Button(description="Generate", button_style="success", icon="music")
out = w.Output()

def post(path, payload):
    req = urllib.request.Request(BASE+path, data=json.dumps(payload).encode(), headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(req, timeout=60) as f:
        return json.loads(f.read())

def unwrap(x):
    return x.get("data", x) if isinstance(x, dict) else x

def slug(s):
    return re.sub(r"[^A-Za-z0-9]+","-",s).strip("-")[:50] or "track"

def auto_title(lyrics, g):
    for line in (lyrics or "").splitlines():
        line = line.strip()
        if line and not line.startswith("["):
            return " ".join(line.split()[:5]).strip(",.!?;:").title()
    return f"{g} Track {datetime.datetime.now().strftime('%H%M')}"

def make_cover(path, g, seed):
    import numpy as np
    from PIL import Image
    rng = np.random.default_rng(seed)
    N = 3000
    y, x = np.mgrid[0:N, 0:N].astype("float32") / N
    f = lambda: rng.uniform(1.5, 5.0)
    v = (np.sin(x*f()*6+y*f()*4+rng.uniform(0,6)) + np.sin(y*f()*7-x*f()*3+rng.uniform(0,6)) + np.sin((x+y)*f()*5+rng.uniform(0,6)))/3
    v = (v+1)/2
    c = np.array(PALETTES[g], dtype="float32")
    t = v[...,None]*2
    lo = np.clip(t,0,1); hi = np.clip(t-1,0,1)
    img = c[0]*(1-lo) + c[1]*(lo*(1-hi)) + c[2]*hi
    Image.fromarray(np.clip(img,0,255).astype("uint8"),"RGB").save(path,"JPEG",quality=92)

def save_release(item, title, g, lang_code, instrumental, artist_name, writer_name, ts):
    lyrics = item.get("lyrics","") or ""
    metas = item.get("metas",{}) or {}
    title = title.strip() or auto_title(lyrics, g)
    folder = f"{OUT_ROOT}/{ts}_{slug(title)}"
    os.makedirs(folder, exist_ok=True)
    raw = f"/content/raw_{ts}.flac"
    urllib.request.urlretrieve(BASE + item["file"], raw)
    base = f"{folder}/{slug(title)}"
    # RouteNote wants stereo, 44.1 kHz, FLAC 16-bit or MP3 320k
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",raw,"-ar","44100","-ac","2","-sample_fmt","s16",base+".flac"],check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",raw,"-ar","44100","-ac","2","-b:a","320k",base+"_320kbps.mp3"],check=True)
    make_cover(folder+"/cover_3000x3000.jpg", g, int(time.time()))
    open(folder+"/lyrics.txt","w").write(lyrics if not instrumental else "[Instrumental]")
    meta = {"title":title,"artist":artist_name,"writer":writer_name,"genre":g,"language":lang_code,
            "instrumental":instrumental,"bpm":metas.get("bpm"),"key":metas.get("keyscale"),
            "duration_sec":metas.get("duration"),"prompt":item.get("prompt"),"seed":item.get("seed_value"),
            "created":ts,"ai_generated":True,
            "models":{"music":item.get("dit_model"),"lm":item.get("lm_model")},
            "licence":"ACE-Step 1.5, MIT licence (commercial use allowed). https://github.com/ace-step/ACE-Step-1.5 and https://huggingface.co/ACE-Step/Ace-Step1.5"}
    json.dump(meta, open(folder+"/metadata.json","w"), indent=2, ensure_ascii=False)
    open(folder+"/routenote_form.txt","w").write(f"""ROUTENOTE RELEASE: copy these into the form (Distribution > Create New Release)
Release title (single = track title): {title}
Artist: {artist_name}
Writer(s) (composer, and lyricist if lyrics): {writer_name}
Primary genre: {g}
Language: {lang_code}
Explicit: check the lyrics, tick Explicit if needed
C line / P line: {artist_name}
Label: {artist_name}
Track audio: {os.path.basename(base)}.flac   (44.1 kHz stereo 16-bit) or the _320kbps.mp3
Artwork: cover_3000x3000.jpg (3000x3000 RGB JPG, no text, no web addresses)
Contributor role: Producer = {artist_name}
Do NOT credit or name the AI company in the release (RouteNote rule).
If RouteNote moderation asks which AI tool you used, tell them: ACE-Step 1.5 (MIT licence)
  https://github.com/ace-step/ACE-Step-1.5
  https://huggingface.co/ACE-Step/Ace-Step1.5
Note: RouteNote cannot send AI-generated tracks to Content ID / Amazon / Korean stores.
Do not describe the track as imitating any real artist.
""")
    newfile = not os.path.exists(OUT_ROOT+"/releases.csv")
    with open(OUT_ROOT+"/releases.csv","a",newline="") as fh:
        cw = csv.writer(fh)
        if newfile: cw.writerow(["created","title","artist","genre","bpm","key","folder"])
        cw.writerow([ts,title,artist_name,g,metas.get("bpm"),metas.get("keyscale"),folder])
    return folder, base, lyrics, metas, title

def generate(payload):
    tid = unwrap(post("/release_task", payload))["task_id"]
    for _i in range(180):
        time.sleep(5)
        res = unwrap(post("/query_result", {"task_id_list":[tid]}))[0]
        if res["status"] == 2:
            raise RuntimeError("Generation failed: " + str(res.get("result"))[:500])
        if res["status"] == 1:
            return json.loads(res["result"])[0]
    raise RuntimeError("Timed out waiting for the model.")

def run(_):
    with out:
        clear_output()
        g = genre.value
        try:
            ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
            payload = {"audio_format":"flac","batch_size":1,"thinking":True,"vocal_language":lang.value,
                       "audio_duration":dur.value,"use_random_seed":True}
            if instr.value:
                payload.update({"prompt":f"{GENRES[g]}. Theme: {idea.value}","lyrics":"[Instrumental]","instrumental":True})
            else:
                payload.update({"sample_mode":True,"sample_query":f"{GENRES[g]}. Song about: {idea.value}"})
            print("Generating... (about 1-4 minutes on a free T4)")
            item = generate(payload)
            folder, base, lyrics, metas, title = save_release(item, title_in.value, g, lang.value, instr.value, artist.value, writer.value, ts)
            clear_output()
            print(f"Saved to Google Drive: My Drive/Zi_Music/{os.path.basename(folder)}")
            print(f"Title: {title} | {g} | BPM {metas.get('bpm')} | key {metas.get('keyscale')}")
            display(Audio(base+"_320kbps.mp3"))
            display(IPImage(folder+"/cover_3000x3000.jpg", width=240))
            print("\n--- LYRICS ---\n" + (lyrics or "[Instrumental]"))
            display(HTML('<p><b>Next:</b> open <a target="_blank" href="https://routenote.com/">RouteNote</a>, log in, Distribution > Create New Release, and copy from <code>routenote_form.txt</code> in that folder.</p>'))
        except Exception as e:
            print("Error:", e)
            print(open('/content/acestep.log').read()[-1500:])

btn.on_click(run)
display(w.VBox([genre, idea, lang, dur, instr, title_in, artist, writer, btn, out]))

## 3) Team mode (optional)
Run cell 2 first. AI employees (lyricist, A&R, producer, engineer, artwork, release manager, operator) make a release with a short log.

In [ ]:
# Presets: sound descriptions only, no artist names.
PRESETS = [
    {
        "name": "Melodic Progressive Techno 122",
        "genre": "melodic progressive techno",
        "bpm": 122,
        "seconds": 120,
        "instrumental": False,
        "idea": "122 BPM melodic progressive techno. Warm analog bassline, rolling kick, lush detuned supersaw pads, plucky arpeggiated lead, soft piano stabs. Mood: hopeful, late-night, driving through city lights. Structure: 16-bar intro (kick + hats), 16-bar build with rising pads, main drop with full melody, short breakdown with piano only, final drop, clean outro for DJ mixing.",
        "lyric_guideline": "Short emotive lines about night drives, city lights, momentum. One 4-line verse, one 2-line hook that repeats. Keep vowels open (ah, oh, ay) so the vocal sits on the melody.",
        "lyric_example": "Verse: Driving through the amber glow / Every light a way to go / Windows down and engines low / Chasing what I used to know. Hook: We ride, we ride into the night / We ride, we ride into the light.",
    },
    {
        "name": "Melodic Progressive Techno 124",
        "genre": "melodic progressive techno",
        "bpm": 124,
        "seconds": 120,
        "instrumental": False,
        "idea": "124 BPM melodic progressive techno. Punchy kick, sidechained bass, hypnotic minor-key arpeggio, wide atmospheric pads, subtle vocal chops. Mood: euphoric but melancholic. Structure: filter-swept intro, groove section, long melodic build, anthemic drop with layered leads, breakdown with arp only, drop reprise, beat-out outro.",
        "lyric_guideline": "Anthem-style hook, max 6 words per line, one line designed to be shouted back. Theme: letting go, weightlessness, summer nights. Verses can be half-spoken.",
        "lyric_example": "Hook: Let it go, let it all go tonight. Verse: Barefoot on the dashboard slow / Radio hums a song we know / Salt air through the window flow / Nowhere left that we won't go.",
    },
    {
        "name": "Indie Dance 118",
        "genre": "indie dance",
        "bpm": 118,
        "seconds": 110,
        "instrumental": False,
        "idea": "118 BPM indie dance / nu-disco. Live-feel bass guitar groove, four-on-the-floor kick, funky rhythm guitar chops, analog synth brass stabs, warm claps. Mood: playful, sun-soaked, slightly retro. Structure: bass-only intro, verse groove, pre-chorus build with guitar, chorus with full band feel, instrumental bridge, double chorus, groove outro.",
        "lyric_guideline": "Conversational, cheeky lyrics. Everyday imagery (coffee, dancing in the kitchen, missed calls). Two short verses, a pre-chorus, and a big repetitive chorus. Keep syllables short for the groove.",
        "lyric_example": "Chorus: Dance with me in the kitchen light / Saturday's ours till Sunday night. Verse: Coffee's cold but I don't mind / Your favorite song is right on time.",
    },
    {
        "name": "Mid-Tempo Techno 110",
        "genre": "mid-tempo techno",
        "bpm": 110,
        "seconds": 100,
        "instrumental": True,
        "idea": "110 BPM mid-tempo techno. Heavy distorted kick, dark rumbling bass, industrial percussion hits, metallic textures, sparse detuned lead motif. Mood: dark warehouse, cinematic tension. Structure: percussive intro, bass drop at bar 17, tension loop with rising noise sweep, stripped middle section, heavy final section with extra percussion layers, sudden cut ending.",
        "lyric_guideline": "Instrumental preset. If vocals wanted, use only a one-word spoken loop (e.g. 'again', 'deeper') pitched down, treated as texture, not a song.",
        "lyric_example": "",
    },
    {
        "name": "Afro-Dance Techno 122",
        "genre": "afro house techno",
        "bpm": 122,
        "seconds": 120,
        "instrumental": False,
        "idea": "122 BPM afro-dance techno. Deep round kick, conga and shaker polyrhythms, log-drum style bass hits, kalimba-like plucks, warm marimba melody, airy pads. Mood: warm, communal, sunset energy. Structure: percussion intro building layer by layer, bass entry, melodic hook section, call-and-response middle, big rhythmic drop, percussion-only outro.",
        "lyric_guideline": "Chant-friendly lines, call and response. One leader line, one crowd answer of 2-4 syllables. Theme: togetherness, dancing till sunrise, home. Repeat more than a normal pop song.",
        "lyric_example": "Call: We came to move. Response: Eh-eh. Call: Under the moon. Response: Eh-eh. Verse: Fire in the drum tonight / Everybody holding tight / From the dust until the light / We keep dancing, it's our right.",
    },
    {
        "name": "Peak Techno 128",
        "genre": "techno",
        "bpm": 128,
        "seconds": 100,
        "instrumental": True,
        "idea": "128 BPM peak-time techno. Hard punchy kick, driving offbeat bass, hypnotic acid-style 303 sequence, sharp hi-hats, white-noise risers. Mood: relentless, focused, peak-hour club. Structure: DJ-friendly 32-bar kick intro, acid line fades in, first drop, breakdown with riser and snare roll, bigger second drop, 32-bar beat outro for mixing.",
        "lyric_guideline": "Instrumental preset. If vocals wanted, a single robotic processed phrase every 8 bars, no melody (e.g. 'don't stop').",
        "lyric_example": "",
    },
    {
        "name": "Melodic Techno 120",
        "genre": "melodic techno",
        "bpm": 120,
        "seconds": 120,
        "instrumental": False,
        "idea": "120 BPM melodic techno. Tight kick, deep sub bass, emotional minor chord progression on a soft pluck, string-like pad swells, gentle arpeggio, distant reverb vocal chops. Mood: introspective, bittersweet, 4am feeling. Structure: ambient pad intro, beat entry, melodic theme, breakdown with strings and spoken feel, emotional final drop, fade outro.",
        "lyric_guideline": "Intimate, near-whispered verses and one soaring hook line. Theme: memory, distance, someone far away. Max 8 words per line, lots of space between phrases.",
        "lyric_example": "Verse: Four a.m., the city's sleeping / Promises we meant for keeping / Half a world between us now / I still hear you, hear you somehow. Hook: Stay with me, even in the silence.",
    },
    {
        "name": "Indie Dance 124 (Night Drive)",
        "genre": "indie dance",
        "bpm": 124,
        "seconds": 120,
        "instrumental": True,
        "idea": "124 BPM indie dance with synthwave edge. Driving bassline, gated reverb snare, neon analog lead, arpeggiated 80s-style synth, soft electric guitar licks. Mood: cinematic night highway, confident, cool. Structure: arp intro, bass and drums entry, lead melody statement, breakdown with guitar licks, final chorus-like section, engine-down outro.",
        "lyric_guideline": "Instrumental preset. Optional: one low spoken hook line per section ('keep driving'), heavy reverb, treated as an instrument.",
        "lyric_example": "",
    },
    {
        "name": "Deep Afro House 121",
        "genre": "afro house",
        "bpm": 121,
        "seconds": 120,
        "instrumental": False,
        "idea": "121 BPM deep afro house. Smooth deep kick, warm sub bass, layered djembe and shaker groove, soulful chord stabs, flute-like lead melody, ocean-breeze pads. Mood: soulful, warm evening, beach bonfire. Structure: percussion and pad intro, bassline entry, vocal verse over groove, melodic hook with flute, breakdown with drums only, full final section, soft outro.",
        "lyric_guideline": "Soulful, simple and warm. Theme: gratitude, home, the sea. One verse, one hook. Long held notes on the hook, short percussive verse lines.",
        "lyric_example": "Verse: Salt on my skin, fire in the sand / Whole wide world in the palm of my hand. Hook: Home is wherever the drums are.",
    },
    {
        "name": "Progressive Build 126",
        "genre": "progressive house techno",
        "bpm": 126,
        "seconds": 120,
        "instrumental": True,
        "idea": "126 BPM progressive house-techno hybrid. Rolling bassline, crisp claps, evolving filter arpeggio that opens slowly, epic layered pads, festival-style riser. Mood: building anticipation, sunrise moment. Structure: long filtered intro, groove lock, 32-bar slow build with opening filters and riser, euphoric drop with full pads and arp, second shorter build and drop, filtered outro.",
        "lyric_guideline": "Instrumental preset. If vocals wanted, one sustained sung note or a single word ('higher') stretched across the build, choir-like layering.",
        "lyric_example": "",
    },
    {
        "name": "Downtempo Indie 100",
        "genre": "downtempo indie electronic",
        "bpm": 100,
        "seconds": 100,
        "instrumental": False,
        "idea": "100 BPM downtempo indie electronic. Soft round kick, warm electric bass, dreamy washed guitars, mellow Rhodes-style keys, gentle synth arpeggio, vinyl-crackle texture. Mood: hazy afternoon, nostalgic, calm. Structure: keys intro, laid-back groove verse, chorus with fuller synths, guitar-texture bridge, final chorus, slow fade.",
        "lyric_guideline": "Soft, half-sung storytelling. Everyday scenes (windows, plants, old photos). Two verses, one gentle chorus, no shouting, no big notes. Lines can be longer and conversational here.",
        "lyric_example": "Verse: Sunday light across the floor / Plants are leaning to the door / Old pictures that I can't ignore / Don't know what I kept them for. Chorus: Slow down, slow down, we got time.",
    },
    {
        "name": "Dark Melodic 123",
        "genre": "dark melodic techno",
        "bpm": 123,
        "seconds": 110,
        "instrumental": True,
        "idea": "123 BPM dark melodic techno. Driving kick, moody minor bassline, haunting detuned lead with long portamento, ominous drone layer, sparse bell-like motif, cinematic percussion. Mood: mysterious, noir, tension with beauty. Structure: drone and percussion intro, bassline entry, lead motif appears, breakdown with bells and drone only, heavy melodic drop, stripped outro.",
        "lyric_guideline": "Instrumental preset. If vocals wanted, whispered single lines, lots of reverb, fragments not full sentences (e.g. 'in the dark we glow').",
        "lyric_example": "",
    },
]


PRESETS += [
 {"name":"Sunrise Desert Drive","genre":"afro melodic house","bpm":121,"seconds":120,"instrumental":False,"key":"D minor",
  "idea":"121 BPM afro melodic house in D minor. Congas, shakers, log drum, deep rolling bass, kalimba-style pluck lead, wide analog pads. Percussion intro, chant vocal texture, call-and-response breakdown, long uplifting drop.",
  "lyric_guideline":"Soulful English vocal about a journey and homecoming. One repeated hook phrase.","lyric_example":""},
 {"name":"Warehouse Afterglow","genre":"deep tech house","bpm":123,"seconds":120,"instrumental":True,"key":"E minor",
  "idea":"123 BPM deep tech house in E minor. Rounded kick, swung sub bass, crisp hats, detuned mid synth stabs, vinyl pad. 32-bar DJ intro and outro, two peaks, filter breakdown.",
  "lyric_guideline":"Mostly instrumental, optional looped spoken phrase.","lyric_example":""},
 {"name":"Hypnotic Coastline","genre":"melodic techno","bpm":122,"seconds":120,"instrumental":True,"key":"A minor",
  "idea":"122 BPM melodic techno in A minor. 16th-note sub bass, minor arpeggio, pad swells, flute motif, rolling toms. Slow two-minute build, arp-only breakdown, powerful re-entry.",
  "lyric_guideline":"Instrumental.","lyric_example":""},
 {"name":"Midnight Bazaar","genre":"afro tech","bpm":122,"seconds":120,"instrumental":False,"key":"F minor",
  "idea":"122 BPM afro tech in F minor. Dark kick, layered hand percussion, marimba-like plucks, sparse vocal chops. Percussion intro, groove locked by bar 32, vocal-chop breakdown.",
  "lyric_guideline":"Sparse short English phrases, chant style.","lyric_example":""},
 {"name":"Neon Mirage","genre":"indie dance dark disco","bpm":118,"seconds":120,"instrumental":False,"key":"G minor",
  "idea":"118 BPM indie dance and dark disco in G minor. Bass guitar groove, analog synth riffs, gated snare, light guitar licks, vocoder double on the chorus.",
  "lyric_guideline":"Cool low English vocal about night drives.","lyric_example":""},
 {"name":"Peak Ritual","genre":"melodic peak techno","bpm":125,"seconds":120,"instrumental":False,"key":"C minor",
  "idea":"125 BPM melodic peak techno in C minor. Driving kick, offbeat bass, euphoric lead motif, risers, long strings and choir breakdown.",
  "lyric_guideline":"Short anthemic original English vocal before each drop.","lyric_example":""},
]

#@title 3) Team mode: AI employees make a release (run cell 2 first)
# Free, open models only. Runs on the free T4. No cards, no paid services.
import json, re, os, time, gc, datetime, traceback, subprocess, urllib.request
import ipywidgets as w
from IPython.display import display, clear_output

LLM_ID = "Qwen/Qwen3-1.7B"   # Apache-2.0 licence (commercial use allowed)
_llm = {}
LOG = []
tlog_out = w.Output()
def log(role, msg):
    line = f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {role}: {msg}"
    LOG.append(line)
    with tlog_out: print(line)

def load_llm():
    if "m" in _llm: return
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    _llm["t"] = AutoTokenizer.from_pretrained(LLM_ID)
    _llm["m"] = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16, device_map="cuda", low_cpu_mem_usage=True)

def free_llm():
    import torch
    _llm.clear(); gc.collect(); torch.cuda.empty_cache()

def ask(system, user, max_new=500, temp=0.8):
    load_llm()
    t, m = _llm["t"], _llm["m"]
    msgs = [{"role":"system","content":system},{"role":"user","content":user}]
    txt = t.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = t(txt, return_tensors="pt").to("cuda")
    o = m.generate(**ids, max_new_tokens=max_new, do_sample=True, temperature=temp, top_p=0.9)
    r = t.decode(o[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", r, flags=re.S).strip()

# ---------- employees ----------
def lyricist(preset, theme, notes=""):
    sys = ("You are a professional songwriter. Write ORIGINAL English song lyrics only. "
           "Use section tags in square brackets: [Verse 1], [Chorus], [Verse 2], [Chorus]. "
           "Short lines, simple words, one repeated hook. Never mention real artists, bands or brands. "
           "Output only the lyrics.")
    usr = f"Style: {preset['genre']}, {preset['bpm']} BPM.\nGuideline: {preset['lyric_guideline']}\nTheme: {theme}\n{notes}"
    return ask(sys, usr, 450)

def ar_review_lyrics(lyrics, preset):
    sys = ("You are a strict A&R reviewer. Score the lyrics 1-10 for: English only, clear hook, "
           "natural rhymes, no clichés overload, no real artist/brand names, fits a dance track. "
           'Reply ONLY with JSON: {"score": n, "fix": "one sentence of what to change"}')
    r = ask(sys, lyrics, 120, 0.2)
    try:
        j = json.loads(re.search(r"\{.*\}", r, re.S).group(0)); return int(j["score"]), j.get("fix","")
    except Exception:
        return 5, "make the hook clearer and the lines shorter"

def english_ok(text):
    letters = [c for c in text if c.isalpha()]
    if not letters: return False
    ascii_ratio = sum(c.isascii() for c in letters)/len(letters)
    common = {"the","and","you","we","to","a","in","my","i","of","on","it","is","me","all","night","love"}
    words = re.findall(r"[a-z']+", text.lower())
    return ascii_ratio > 0.97 and sum(x in common for x in words) >= 5

def producer(preset, bpm, theme, instrumental):
    key = preset.get("key", "")
    cap = preset["idea"] + (f" Key: {key}." if key else "") + f" Theme: {theme}."
    return {"caption": cap, "bpm": bpm, "instrumental": instrumental}

def audio_check(path, seconds):
    # Objective A&R checks with ffmpeg: duration, silence, clipping.
    try:
        dur = float(subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",path],
                    capture_output=True, text=True).stdout.strip())
        st = subprocess.run(["ffmpeg","-i",path,"-af","volumedetect","-f","null","-"], capture_output=True, text=True).stderr
        mv = re.search(r"mean_volume: (-?[\d.]+) dB", st); mx = re.search(r"max_volume: (-?[\d.]+) dB", st)
        mean = float(mv.group(1)) if mv else -99; peak = float(mx.group(1)) if mx else -99
        ok = dur >= seconds*0.8 and mean > -35 and peak < 0.5
        return ok, f"duration {dur:.0f}s, mean {mean:.1f} dB, peak {peak:.1f} dB"
    except Exception as e:
        return False, f"check failed: {e}"

def operator_health():
    # Operator: checks the music server and GPU before work.
    try:
        urllib.request.urlopen(BASE + "/health", timeout=10).read(); return True, "server ok"
    except Exception as e:
        try:
            urllib.request.urlopen(BASE, timeout=10); return True, "server answers"
        except Exception as e2:
            return False, f"music server not answering ({str(e2)[:80]}). Re-run cell 1 (setup), then retry."

def explain_error(e):
    s = str(e)
    if "NaN" in s: return "float16 overflow on the T4. The fp32 patch in cell 1 must be applied. Re-run cell 1."
    if "out of memory" in s.lower(): return "GPU memory full. Using shorter length or restarting the runtime fixes it."
    if "Timed out" in s: return "the model took too long (first run loads models). Run again."
    return s[:200]

PRESET_BY_NAME = {p["name"]: p for p in PRESETS}
for _p in PRESETS:
    PALETTES.setdefault(_p["genre"], PALETTES["Techno"])
p_dd = w.Dropdown(options=list(PRESET_BY_NAME), description="Preset", layout=w.Layout(width="420px"))
bpm_dd = w.Dropdown(options=[("Use preset BPM", 0), ("120", 120), ("124", 124), ("128", 128)], description="BPM")
t_theme = w.Text(value="a late night drive through the city", description="Theme", layout=w.Layout(width="520px"))
t_instr = w.Dropdown(options=[("Preset default", "d"), ("Vocals (English)", "v"), ("Instrumental", "i")], description="Vocals")
t_secs = w.IntSlider(value=90, min=60, max=180, step=10, description="Seconds")
t_title = w.Text(placeholder="leave empty = auto title", description="Title", layout=w.Layout(width="520px"))
t_artist = w.Text(value="Rahal", placeholder="your artist name", description="Artist", layout=w.Layout(width="520px"))
t_writer = w.Text(value="Rahal", placeholder="your legal first + last name", description="Writer", layout=w.Layout(width="520px"))
t_btn = w.Button(description="Run the team", button_style="primary", icon="users")

def run_team(_):
    with tlog_out: clear_output()
    LOG.clear()
    pr = PRESET_BY_NAME[p_dd.value]
    bpm = bpm_dd.value or pr["bpm"]
    instrumental = pr["instrumental"] if t_instr.value == "d" else (t_instr.value == "i")
    ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    try:
        log("Operator", "starting job, checking health")
        ok, msg = operator_health(); log("Operator", msg)
        if not ok: return
        lyrics = "[Instrumental]"
        if not instrumental:
            best, best_s = None, -1
            for attempt in range(1, 4):
                log("Lyricist", f"writing draft {attempt}")
                d = lyricist(pr, t_theme.value, "" if attempt == 1 else f"Previous fix request: {fix}")
                if not english_ok(d):
                    log("A&R", "rejected: not clean English"); fix = "write plain English only"; continue
                s, fix = ar_review_lyrics(d, pr)
                log("A&R", f"lyrics score {s}/10. {fix}")
                if s > best_s: best, best_s = d, s
                if s >= 7: break
            if best is None:
                log("Operator", "no acceptable English lyrics. Falling back to instrumental."); instrumental = True
            else:
                lyrics = best
            free_llm(); log("Operator", "freed GPU memory for the music model")
        else:
            log("Lyricist", "skipped (instrumental)")
        plan = producer(pr, bpm, t_theme.value, instrumental)
        log("Producer", f"{bpm} BPM, {pr['genre']}, key {pr.get('key','auto')}")
        payload = {"audio_format":"flac","batch_size":1,"thinking":False,"vocal_language":"en","audio_duration":t_secs.value,
                   "use_random_seed":True,"prompt":plan["caption"],"lyrics":lyrics,"instrumental":instrumental,"bpm":bpm}
        if pr.get("key"): payload["key_scale"] = pr["key"]
        item = None
        for attempt in range(1, 3):
            log("Engineer", f"rendering audio (attempt {attempt}), 1-4 min")
            try:
                item = generate(payload)
            except Exception as e:
                log("Operator", "error: " + explain_error(e)); item = None; continue
            raw = f"/content/raw_{ts}_{attempt}.flac"
            urllib.request.urlretrieve(BASE + item["file"], raw)
            good, info = audio_check(raw, t_secs.value)
            log("A&R", ("approved: " if good else "rejected: ") + info)
            if good: break
            item = None
        if item is None:
            log("Operator", "no track passed review. Nothing saved. See the notes above."); return
        title = t_title.value
        if not title.strip():
            title = (re.sub(r"[^A-Za-z ]","",pr["name"]) + " " + ts[-4:]).strip()
        log("Artwork+Metadata", "making 3000x3000 cover, metadata.json, FLAC and MP3 320k")
        item["lyrics"] = lyrics if not instrumental else ""
        item.setdefault("metas", {}); item["metas"].update({"bpm": bpm, "keyscale": pr.get("key")})
        folder, base, _l, metas, title = save_release(item, title, pr["genre"], "en", instrumental,
                                                      t_artist.value or "Artist", t_writer.value or "Writer", ts)
        open(folder + "/team_log.txt", "w").write("\n".join(LOG))
        log("Release manager", f"RouteNote sheet and releases.csv updated: My Drive/Zi_Music/{os.path.basename(folder)}")
        display(Audio(base + ".flac"))
    except Exception as e:
        log("Operator", "unexpected error: " + explain_error(e))
        with tlog_out: print(traceback.format_exc()[-600:])

t_btn.on_click(run_team)
display(w.VBox([p_dd, bpm_dd, t_theme, t_instr, t_secs, t_title, t_artist, t_writer, t_btn, tlog_out]))


In [ ]:
#@title 4) Batch mode: make 10 tracks back to back (run cells 2 and 3 first)
# RAM-safe order: all lyrics first (small model on GPU), free it, then render all audio.
import datetime, os, re, urllib.request, traceback
import ipywidgets as w
from IPython.display import display
b_n = w.IntSlider(value=10, min=1, max=18, description="Tracks")
b_secs = w.IntSlider(value=90, min=60, max=180, step=10, description="Seconds")
b_artist = w.Text(value="Rahal", placeholder="your artist name", description="Artist", layout=w.Layout(width="520px"))
b_writer = w.Text(value="Rahal", placeholder="your legal first + last name", description="Writer", layout=w.Layout(width="520px"))
b_btn = w.Button(description="Make the batch", button_style="warning", icon="rocket")
b_out = w.Output()
THEMES = ["a late night drive through the city","dancing until sunrise","leaving something behind","summer by the sea",
          "city lights from a rooftop","coming home after a long trip","a secret meeting at midnight","the desert at dawn",
          "feeling weightless","a train crossing the country","waiting for a message","golden hour on the water"]
def blog(role, msg):
    with b_out: print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {role}: {msg}")
def run_batch(_):
    with b_out: clear_output()
    jobs = []
    for i in range(b_n.value):
        pr = PRESETS[i % len(PRESETS)]
        jobs.append({"pr": pr, "theme": THEMES[i % len(THEMES)], "instr": pr["instrumental"], "lyrics": "[Instrumental]"})
    try:
        ok, msg = operator_health(); blog("Operator", msg)
        if not ok: return
        # phase 1: lyrics
        for n, j in enumerate(jobs, 1):
            if j["instr"]: continue
            best, best_s, fix = None, -1, ""
            for attempt in range(2):
                try:
                    d = lyricist(j["pr"], j["theme"], f"Fix request: {fix}" if fix else "")
                except Exception as e:
                    blog("Operator", "lyricist error: " + explain_error(e)); break
                if not english_ok(d): fix = "plain English only"; continue
                s, fix = ar_review_lyrics(d, j["pr"])
                if s > best_s: best, best_s = d, s
                if s >= 7: break
            if best is None:
                j["instr"] = True; blog("A&R", f"#{n} {j['pr']['name']}: no good English lyrics, making it instrumental")
            else:
                j["lyrics"] = best; blog("A&R", f"#{n} {j['pr']['name']}: lyrics {best_s}/10")
        try: free_llm()
        except Exception: pass
        blog("Operator", "lyrics done, GPU freed, rendering audio")
        # phase 2: audio
        done = 0
        for n, j in enumerate(jobs, 1):
            pr = j["pr"]; ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
            try:
                payload = {"audio_format":"flac","batch_size":1,"thinking":False,"vocal_language":"en","audio_duration":b_secs.value,
                           "use_random_seed":True,"prompt":pr["idea"] + f" Theme: {j['theme']}.","lyrics":j["lyrics"],
                           "instrumental":j["instr"],"bpm":pr["bpm"]}
                if pr.get("key"): payload["key_scale"] = pr["key"]
                item = None
                for attempt in (1, 2):
                    it = generate(payload)
                    raw = f"/content/raw_{ts}_{attempt}.flac"
                    urllib.request.urlretrieve(BASE + it["file"], raw)
                    good, info = audio_check(raw, b_secs.value)
                    blog("A&R", f"#{n} {pr['name']} attempt {attempt}: {'approved' if good else 'rejected'} ({info})")
                    if good: item = it; break
                if item is None: continue
                it["lyrics"] = "" if j["instr"] else j["lyrics"]
                it.setdefault("metas", {}); it["metas"].update({"bpm": pr["bpm"], "keyscale": pr.get("key")})
                title = (re.sub(r"[^A-Za-z ]", "", pr["name"]) + " " + ts[-4:]).strip()
                folder, base, _l, metas, title = save_release(it, title, pr["genre"], "en", j["instr"],
                                                              b_artist.value or "Artist", b_writer.value or "Writer", ts)
                done += 1; blog("Release manager", f"#{n} saved: {os.path.basename(folder)}")
            except Exception as e:
                blog("Operator", f"#{n} {pr['name']} failed: {explain_error(e)}. Skipping to the next track.")
                ok, msg = operator_health()
                if not ok: blog("Operator", msg); break
        blog("Release manager", f"batch finished: {done}/{len(jobs)} tracks in My Drive/Zi_Music. Open releases.csv for the list.")
    except Exception as e:
        blog("Operator", "unexpected error: " + explain_error(e))
        with b_out: print(traceback.format_exc()[-600:])
b_btn.on_click(run_batch)
display(w.VBox([b_n, b_secs, b_artist, b_writer, b_btn, b_out]))


## Notes
- **Free limits:** Colab free GPU sessions close after a few hours or when idle. Run cell 1 again after that. Your songs stay in Drive.
- **Upload to RouteNote:** RouteNote has no public upload API, so this notebook prepares everything (audio at 44.1 kHz, 3000x3000 cover, metadata sheet). You log in to RouteNote and upload; each folder contains `routenote_form.txt` to copy from.
- **RouteNote and AI music:** accepted with conditions (no imitation of real artists, do not credit the AI company, no Content ID / Amazon / Korean stores, extra moderation time).
- **Licence:** ACE-Step 1.5 is MIT. Check the current model page before each release batch.